# Results Analysis & Paper Figures Generation

**Purpose:** Evaluate all trained models and generate publication-ready results

## Generates:
- ✅ **Table 1:** Main results (BLEU, chrF++, Length Ratio)
- ✅ **Table 2:** Dataset statistics
- ✅ **Table 3-4:** Error analysis (terminology, modality)
- ✅ **Figure 1:** Corpus length distribution
- ✅ **Figure 2:** Model comparison (BLEU scores)
- ✅ **Figure 3:** Directional asymmetry (**KEY FINDING**)
- ✅ **Figure 4:** Error analysis visualizations
- ✅ **Table 5:** Translation examples (best & worst)

## Prerequisites:
1. Trained models in `/content/models/` (from training notebooks)
2. Test data in `/MyDrive/Legal_NLP/data/splits/`
3. A100 GPU recommended for fast inference

**All results saved to:** `/MyDrive/Legal_NLP/results/`

---

## Cell 1: Setup

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
print("Installing packages...")
!pip install -q torch transformers sacrebleu evaluate
!pip install -q pandas numpy matplotlib seaborn plotly
!pip install -q scikit-learn tqdm

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from tqdm import tqdm
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MBart50TokenizerFast
from sacrebleu.metrics import BLEU, CHRF

# Paths
DRIVE_BASE = '/content/drive/MyDrive/Legal_NLP'

# Create output directories
os.makedirs(f"{DRIVE_BASE}/results/tables", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/results/figures", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/results/translations", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/results/error_analysis", exist_ok=True)

# Set plot style
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['figure.dpi'] = 300

print(f"\n✅ Setup complete!")
print(f"Results will be saved to: {DRIVE_BASE}/results/")

## Cell 2: Generate Table 2 - Dataset Statistics

In [ ]:
# Load all splits
train_df = pd.read_csv(f'{DRIVE_BASE}/data/splits/train.csv')
val_df = pd.read_csv(f'{DRIVE_BASE}/data/splits/val.csv')
test_df = pd.read_csv(f'{DRIVE_BASE}/data/splits/test.csv')

# Combine for overall statistics
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Compute statistics
df['en_len'] = df['English'].str.split().str.len()
df['ne_len'] = df['Nepali'].str.split().str.len()

stats = {
    'Statistic': [
        'Total Pairs',
        'Train / Val / Test',
        'Avg English Length',
        'Avg Nepali Length',
        'Max English Length',
        'Max Nepali Length',
        'Min English Length',
        'Min Nepali Length'
    ],
    'Value': [
        f"{len(df)}",
        f"{len(train_df)} / {len(val_df)} / {len(test_df)}",
        f"{df['en_len'].mean():.2f}",
        f"{df['ne_len'].mean():.2f}",
        f"{df['en_len'].max()}",
        f"{df['ne_len'].max()}",
        f"{df['en_len'].min()}",
        f"{df['ne_len'].min()}"
    ]
}

table2 = pd.DataFrame(stats)

print("="*80)
print("TABLE 2: DATASET STATISTICS")
print("="*80)
print(table2.to_string(index=False))
print("="*80)

# Save
table2.to_csv(f"{DRIVE_BASE}/results/tables/table2_dataset_statistics.csv", index=False)
print(f"\n✅ Saved to: {DRIVE_BASE}/results/tables/table2_dataset_statistics.csv")

## Cell 3: Generate Figure 1 - Corpus Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# English
axes[0].hist(df['en_len'], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[0].axvline(df['en_len'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {df["en_len"].mean():.1f}')
axes[0].set_xlabel('Sentence Length (words)', fontweight='bold')
axes[0].set_ylabel('Frequency', fontweight='bold')
axes[0].set_title('English Sentence Length Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Nepali
axes[1].hist(df['ne_len'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1].axvline(df['ne_len'].mean(), color='blue', linestyle='--', linewidth=2,
                label=f'Mean: {df["ne_len"].mean():.1f}')
axes[1].set_xlabel('Sentence Length (words)', fontweight='bold')
axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].set_title('Nepali Sentence Length Distribution', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/results/figures/figure1_corpus_length_distribution.png',
            dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure 1 saved: figure1_corpus_length_distribution.png")

## Cell 4: Evaluation Function

In [ ]:
def translate_batch(texts, model, tokenizer, src_lang, tgt_lang, model_type,
                    batch_size=16, max_length=128, num_beams=4):
    """
    Translate texts in batches (sorted by length for efficiency).
    Based on working NLLB example.
    """
    # Sort by length for efficient batching
    idxs, texts_sorted = zip(*sorted(enumerate(texts), key=lambda p: len(p[1]), reverse=True))
    results = []

    for i in tqdm(range(0, len(texts_sorted), batch_size), desc="Translating"):
        batch = texts_sorted[i:i+batch_size]

        # Set source language
        if hasattr(tokenizer, 'src_lang'):
            tokenizer.src_lang = src_lang
        if hasattr(tokenizer, 'tgt_lang') and model_type == 'mbart':
            tokenizer.tgt_lang = tgt_lang

        # Tokenize
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=max_length
        ).to(model.device)

        # Generate
        with torch.no_grad():
            if model_type == 'nllb':
                # NLLB requires forced_bos_token_id
                tgt_id = tokenizer.convert_tokens_to_ids(tgt_lang)
                outputs = model.generate(
                    **inputs,
                    forced_bos_token_id=tgt_id,
                    max_new_tokens=int(32 + 3 * inputs.input_ids.shape[1]),  # Dynamic length
                    num_beams=num_beams
                )
            else:
                # mBART
                outputs = model.generate(
                    **inputs,
                    max_length=max_length,
                    num_beams=num_beams
                )

        # Decode
        batch_results = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(batch_results)

    # Restore original order
    return [result for _, result in sorted(zip(idxs, results))]


def evaluate_model(model_path, model_type, direction, test_data_path, output_dir):
    """
    Evaluate a trained model on test set using proper inference.

    Returns:
        dict: BLEU, chrF++, length_ratio scores and translations
    """
    print(f"\n{'='*80}")
    print(f"Evaluating: {model_type.upper()} - {direction.upper()}")
    print("="*80)

    # Load tokenizer and model
    if model_type == 'mbart':
        tokenizer = MBart50TokenizerFast.from_pretrained(model_path)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_path)

    model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    model = model.to('cuda')
    model.eval()

    print(f"✅ Loaded model from: {model_path}")

    # Load test data
    test_df = pd.read_csv(test_data_path)

    # Set language codes and columns
    if direction == 'ne_en':
        src_col, tgt_col = 'Nepali', 'English'
        if model_type == 'mbart':
            src_lang, tgt_lang = 'ne_NP', 'en_XX'
        else:
            src_lang, tgt_lang = 'npi_Deva', 'eng_Latn'
    else:  # en_ne
        src_col, tgt_col = 'English', 'Nepali'
        if model_type == 'mbart':
            src_lang, tgt_lang = 'en_XX', 'ne_NP'
        else:
            src_lang, tgt_lang = 'eng_Latn', 'npi_Deva'

    sources = test_df[src_col].tolist()
    references = test_df[tgt_col].tolist()

    # Translate using batched function (like working example)
    print("Translating test set...")
    hypotheses = translate_batch(
        sources, model, tokenizer, src_lang, tgt_lang, model_type,
        batch_size=16, num_beams=4
    )

    # Compute metrics using sacrebleu (like working example)
    bleu_calc = BLEU()
    chrf_calc = CHRF(word_order=2)  # chrF++

    # sacrebleu expects references as list of lists
    refs = [[r] for r in references]

    bleu_score = bleu_calc.corpus_score(hypotheses, refs)
    chrf_score = chrf_calc.corpus_score(hypotheses, refs)

    # Length ratio
    hyp_lens = [len(h.split()) for h in hypotheses]
    ref_lens = [len(r.split()) for r in references]
    length_ratio = np.mean(hyp_lens) / np.mean(ref_lens)

    results = {
        'model_type': model_type,
        'direction': direction,
        'bleu': bleu_score.score,
        'chrf++': chrf_score.score,
        'length_ratio': length_ratio
    }

    print(f"\nResults:")
    print(f"  BLEU: {results['bleu']:.2f}")
    print(f"  chrF++: {results['chrf++']:.2f}")
    print(f"  Length Ratio: {results['length_ratio']:.3f}")

    # Save results
    os.makedirs(output_dir, exist_ok=True)

    with open(f"{output_dir}/metrics.json", 'w') as f:
        json.dump(results, f, indent=2)

    pd.DataFrame({
        'source': sources,
        'reference': references,
        'hypothesis': hypotheses
    }).to_csv(f"{output_dir}/translations.csv", index=False)

    print(f"✅ Results saved to: {output_dir}")
    return results, hypotheses

print("✅ Evaluation functions defined (using working NLLB inference approach)")

## Cell 5: Evaluate All Models (Table 1)

In [ ]:
test_path = f'{DRIVE_BASE}/data/splits/test.csv'
all_results = {}

# Define models to evaluate
models_to_eval = [
    ('mbart', 'ne_en', '/content/models/mbart50_ne_en/final_model', 'uni'),
    ('mbart', 'en_ne', '/content/models/mbart50_en_ne/final_model', 'uni'),
    ('mbart', 'ne_en', '/content/models/mbart50_bidirectional/final_model', 'bi'),
    ('mbart', 'en_ne', '/content/models/mbart50_bidirectional/final_model', 'bi'),
    ('nllb', 'ne_en', '/content/models/nllb200_ne_en/final_model', 'uni'),
    ('nllb', 'en_ne', '/content/models/nllb200_en_ne/final_model', 'uni'),
    ('nllb', 'ne_en', '/content/models/nllb200_bidirectional/final_model', 'bi'),
    ('nllb', 'en_ne', '/content/models/nllb200_bidirectional/final_model', 'bi'),
]

for model_type, direction, model_path, config in models_to_eval:
    if os.path.exists(model_path):
        output_dir = f"{DRIVE_BASE}/results/translations/{model_type}_{config}_{direction}"
        results, _ = evaluate_model(model_path, model_type, direction, test_path, output_dir)
        all_results[f"{model_type}_{config}_{direction}"] = results
    else:
        print(f"⚠️ Model not found: {model_path}")
        print(f"   Skipping {model_type}_{config}_{direction}")

# Save all results
with open(f"{DRIVE_BASE}/results/all_results.json", 'w') as f:
    json.dump(all_results, f, indent=2)

# Create Table 1
summary = []
for key, res in all_results.items():
    config_type = 'Bi-directional' if '_bi_' in key else 'Uni-directional'
    summary.append({
        'Model': res['model_type'].upper(),
        'Configuration': config_type,
        'Direction': res['direction'].replace('_', '→').upper(),
        'BLEU': f"{res['bleu']:.2f}",
        'chrF++': f"{res['chrf++']:.2f}",
        'Length Ratio': f"{res['length_ratio']:.3f}"
    })

table1 = pd.DataFrame(summary)

print("\n" + "="*80)
print("TABLE 1: MAIN RESULTS (For Paper)")
print("="*80)
print(table1.to_string(index=False))
print("="*80)

table1.to_csv(f"{DRIVE_BASE}/results/tables/table1_main_results.csv", index=False)
print(f"\n✅ Table 1 saved: table1_main_results.csv")

## Cell 6: Generate Figure 2 - Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

labels = []
bleu_scores = []
colors = []

for key, res in all_results.items():
    config = 'Bi-dir' if '_bi_' in key else 'Uni-dir'
    label = f"{res['model_type'].upper()}\n{res['direction'].replace('_','→').upper()}\n({config})"
    labels.append(label)
    bleu_scores.append(res['bleu'])
    colors.append('#e74c3c' if '_bi_' in key else '#3498db')

x = np.arange(len(labels))
bars = ax.bar(x, bleu_scores, color=colors, alpha=0.8, edgecolor='black')

ax.set_xlabel('Model Configuration', fontweight='bold', fontsize=12)
ax.set_ylabel('BLEU Score', fontweight='bold', fontsize=12)
ax.set_title('BLEU Score Comparison Across All Configurations', 
             fontweight='bold', fontsize=14, pad=20)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='Uni-directional'),
    Patch(facecolor='#e74c3c', label='Bi-directional')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/results/figures/figure2_bleu_comparison.png',
            dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure 2 saved: figure2_bleu_comparison.png")

## Cell 7: Generate Figure 3 - Directional Asymmetry (KEY FINDING)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, model_type in enumerate(['mbart', 'nllb']):
    # Get results
    ne_en_uni = all_results[f"{model_type}_uni_ne_en"]
    en_ne_uni = all_results[f"{model_type}_uni_en_ne"]
    ne_en_bi = all_results[f"{model_type}_bi_ne_en"]
    en_ne_bi = all_results[f"{model_type}_bi_en_ne"]

    metrics = ['BLEU', 'chrF++']
    ne_en_scores = [ne_en_uni['bleu'], ne_en_uni['chrf++']]
    en_ne_scores_uni = [en_ne_uni['bleu'], en_ne_uni['chrf++']]
    en_ne_scores_bi = [en_ne_bi['bleu'], en_ne_bi['chrf++']]

    x = np.arange(len(metrics))
    width = 0.25

    axes[idx].bar(x - width, ne_en_scores, width, label='NE→EN (Uni)', 
                 color='#2ecc71', alpha=0.8)
    axes[idx].bar(x, en_ne_scores_uni, width, label='EN→NE (Uni)', 
                 color='#3498db', alpha=0.8)
    axes[idx].bar(x + width, en_ne_scores_bi, width, label='EN→NE (Bi)', 
                 color='#e74c3c', alpha=0.8)

    axes[idx].set_xlabel('Metric', fontweight='bold')
    axes[idx].set_ylabel('Score', fontweight='bold')
    axes[idx].set_title(f'{model_type.upper()} Directional Asymmetry', fontweight='bold')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(metrics)
    axes[idx].legend()
    axes[idx].grid(axis='y', alpha=0.3, linestyle='--')

    # Highlight degradation
    degradation = en_ne_uni['bleu'] - en_ne_bi['bleu']
    axes[idx].text(0.5, 0.95, f'EN→NE Degradation: {degradation:.2f} BLEU',
                  transform=axes[idx].transAxes, ha='center', va='top',
                  bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5),
                  fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/results/figures/figure3_directional_asymmetry.png',
            dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure 3 saved: figure3_directional_asymmetry.png")
print("   ⚠️ This is your KEY FINDING for the paper!")

## Cell 8: Error Analysis Function

In [ ]:
def analyze_errors(translations_path, direction):
    """
    Comprehensive error analysis for legal translation.
    """
    df = pd.read_csv(translations_path)

    # Legal terminology
    legal_terms = {
        'constitution', 'law', 'statute', 'regulation', 'provision',
        'article', 'section', 'clause', 'shall', 'must', 'may',
        'court', 'justice', 'rights', 'duty', 'obligation'
    }

    # Modal verbs
    modals = {'shall', 'must', 'may', 'should', 'can', 'will'}

    # Analyze terminology preservation
    term_preservation = []
    for _, row in df.iterrows():
        ref_terms = set([w.lower() for w in row['reference'].split() if w.lower() in legal_terms])
        hyp_terms = set([w.lower() for w in row['hypothesis'].split() if w.lower() in legal_terms])
        if ref_terms:
            preservation = len(ref_terms & hyp_terms) / len(ref_terms)
            term_preservation.append(preservation)

    # Analyze modal preservation
    modal_preservation = []
    for _, row in df.iterrows():
        ref_modals = [m for m in modals if m in row['reference'].lower().split()]
        hyp_modals = [m for m in modals if m in row['hypothesis'].lower().split()]
        if ref_modals:
            preservation = len(set(ref_modals) & set(hyp_modals)) / len(set(ref_modals))
            modal_preservation.append(preservation)

    # Length analysis
    df['ref_len'] = df['reference'].str.split().str.len()
    df['hyp_len'] = df['hypothesis'].str.split().str.len()
    df['length_ratio'] = df['hyp_len'] / df['ref_len']

    too_short = len(df[df['length_ratio'] < 0.5])
    too_long = len(df[df['length_ratio'] > 2.0])
    appropriate = len(df) - too_short - too_long

    results = {
        'direction': direction,
        'terminology': {
            'avg_preservation': np.mean(term_preservation) if term_preservation else 0,
            'perfect_preservation_rate': sum(1 for x in term_preservation if x == 1.0) / len(term_preservation) if term_preservation else 0,
            'examples_with_terms': len(term_preservation)
        },
        'modality': {
            'avg_preservation': np.mean(modal_preservation) if modal_preservation else 0,
            'perfect_preservation_rate': sum(1 for x in modal_preservation if x == 1.0) / len(modal_preservation) if modal_preservation else 0,
            'examples_with_modals': len(modal_preservation)
        },
        'length': {
            'too_short': too_short,
            'appropriate': appropriate,
            'too_long': too_long,
            'too_short_pct': too_short / len(df) * 100,
            'appropriate_pct': appropriate / len(df) * 100,
            'too_long_pct': too_long / len(df) * 100
        }
    }

    return results

print("✅ Error analysis function defined")

## Cell 9: Generate Tables 3 & 4 - Error Analysis

In [ ]:
error_results = {}

for key in all_results.keys():
    parts = key.split('_')
    model_type = parts[0]
    config = parts[1]
    direction = parts[2] + '_' + parts[3]

    trans_path = f"{DRIVE_BASE}/results/translations/{key}/translations.csv"
    if os.path.exists(trans_path):
        results = analyze_errors(trans_path, direction)
        error_results[key] = results

        # Save individual analysis
        with open(f"{DRIVE_BASE}/results/error_analysis/{key}_errors.json", 'w') as f:
            json.dump(results, f, indent=2)

# Create Tables 3 & 4
term_data = []
modal_data = []

for key, res in error_results.items():
    parts = key.split('_')
    model = parts[0].upper()
    config = 'Bi-directional' if parts[1] == 'bi' else 'Uni-directional'
    direction = (parts[2] + '→' + parts[3]).upper()

    term_data.append({
        'Model': model,
        'Config': config,
        'Direction': direction,
        'Avg Preservation': f"{res['terminology']['avg_preservation']:.2%}",
        'Perfect Preservation': f"{res['terminology']['perfect_preservation_rate']:.2%}"
    })

    modal_data.append({
        'Model': model,
        'Config': config,
        'Direction': direction,
        'Avg Preservation': f"{res['modality']['avg_preservation']:.2%}",
        'Perfect Preservation': f"{res['modality']['perfect_preservation_rate']:.2%}"
    })

table3 = pd.DataFrame(term_data)
table4 = pd.DataFrame(modal_data)

print("\n" + "="*80)
print("TABLE 3: TERMINOLOGY PRESERVATION")
print("="*80)
print(table3.to_string(index=False))

print("\n" + "="*80)
print("TABLE 4: MODALITY PRESERVATION")
print("="*80)
print(table4.to_string(index=False))
print("="*80)

table3.to_csv(f"{DRIVE_BASE}/results/tables/table3_terminology.csv", index=False)
table4.to_csv(f"{DRIVE_BASE}/results/tables/table4_modality.csv", index=False)

print("\n✅ Tables 3 & 4 saved")

## Cell 10: Generate Figure 4 - Error Analysis Visualizations

In [ ]:
# Use mBART NE→EN as example
key = 'mbart_uni_ne_en'
if key in error_results:
    res = error_results[key]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Terminology preservation
    term_metrics = ['Avg\nPreservation', 'Perfect\nPreservation']
    term_values = [
        res['terminology']['avg_preservation'],
        res['terminology']['perfect_preservation_rate']
    ]
    axes[0, 0].bar(term_metrics, term_values, color=['#3498db', '#2ecc71'], alpha=0.7)
    axes[0, 0].set_ylabel('Rate', fontweight='bold')
    axes[0, 0].set_title('Legal Terminology Preservation', fontweight='bold')
    axes[0, 0].set_ylim([0, 1])
    axes[0, 0].grid(axis='y', alpha=0.3)

    # 2. Modality preservation
    modal_metrics = ['Avg\nPreservation', 'Perfect\nPreservation']
    modal_values = [
        res['modality']['avg_preservation'],
        res['modality']['perfect_preservation_rate']
    ]
    axes[0, 1].bar(modal_metrics, modal_values, color=['#e74c3c', '#f39c12'], alpha=0.7)
    axes[0, 1].set_ylabel('Rate', fontweight='bold')
    axes[0, 1].set_title('Modal Verb Preservation', fontweight='bold')
    axes[0, 1].set_ylim([0, 1])
    axes[0, 1].grid(axis='y', alpha=0.3)

    # 3. Length appropriateness pie chart
    length_labels = ['Too Short', 'Appropriate', 'Too Long']
    length_values = [
        res['length']['too_short'],
        res['length']['appropriate'],
        res['length']['too_long']
    ]
    colors_pie = ['#e74c3c', '#2ecc71', '#f39c12']
    axes[1, 0].pie(length_values, labels=length_labels, colors=colors_pie,
                   autopct='%1.1f%%', startangle=90)
    axes[1, 0].set_title('Length Appropriateness Distribution', fontweight='bold')

    # 4. Summary text
    axes[1, 1].axis('off')
    summary_text = f"""Error Analysis Summary

Model: {key.upper()}

Terminology:
  • Avg preservation: {res['terminology']['avg_preservation']:.2%}
  • Examples analyzed: {res['terminology']['examples_with_terms']}

Modality:
  • Avg preservation: {res['modality']['avg_preservation']:.2%}
  • Examples analyzed: {res['modality']['examples_with_modals']}

Length:
  • Too short: {res['length']['too_short_pct']:.1f}%
  • Appropriate: {res['length']['appropriate_pct']:.1f}%
  • Too long: {res['length']['too_long_pct']:.1f}%
    """
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                   verticalalignment='center')

    plt.tight_layout()
    plt.savefig(f'{DRIVE_BASE}/results/figures/figure4_error_analysis.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    print("✅ Figure 4 saved: figure4_error_analysis.png")
else:
    print("⚠️ Error results not found")

## Cell 11: Generate Table 5 - Translation Examples

In [ ]:
# Use mBART NE→EN translations
trans_path = f"{DRIVE_BASE}/results/translations/mbart_uni_ne_en/translations.csv"

if os.path.exists(trans_path):
    trans_df = pd.read_csv(trans_path)

    # Calculate sentence-level BLEU scores
    bleu = BLEU()
    scores = []
    for _, row in trans_df.iterrows():
        score = bleu.sentence_score(row['hypothesis'], [row['reference']])
        scores.append(score.score)

    trans_df['bleu_score'] = scores

    # Get best and worst examples
    best_5 = trans_df.nlargest(5, 'bleu_score')[['source', 'reference', 'hypothesis', 'bleu_score']]
    worst_5 = trans_df.nsmallest(5, 'bleu_score')[['source', 'reference', 'hypothesis', 'bleu_score']]

    print("="*80)
    print("TABLE 5: TRANSLATION EXAMPLES (For Paper)")
    print("="*80)
    print("\nBEST TRANSLATIONS:")
    print(best_5.to_string(index=False))
    print("\nWORST TRANSLATIONS:")
    print(worst_5.to_string(index=False))
    print("="*80)

    # Save
    best_5.to_csv(f"{DRIVE_BASE}/results/tables/table5_examples_best.csv", index=False)
    worst_5.to_csv(f"{DRIVE_BASE}/results/tables/table5_examples_worst.csv", index=False)

    print("\n✅ Table 5 saved (best & worst examples)")
else:
    print("⚠️ Translation file not found")

## Cell 12: Summary & Package Results

In [ ]:
import shutil
from datetime import datetime

print("="*80)
print("RESULTS GENERATION COMPLETE")
print("="*80)

print("\n✅ TABLES GENERATED:")
print("  • Table 1: Main results (BLEU, chrF++)")
print("  • Table 2: Dataset statistics")
print("  • Table 3: Terminology preservation")
print("  • Table 4: Modality preservation")
print("  • Table 5: Translation examples")

print("\n✅ FIGURES GENERATED:")
print("  • Figure 1: Corpus length distribution")
print("  • Figure 2: BLEU comparison")
print("  • Figure 3: Directional asymmetry (KEY FINDING)")
print("  • Figure 4: Error analysis")

print("\n✅ ALL RESULTS SAVED TO:")
print(f"  {DRIVE_BASE}/results/")
print("\n  Structure:")
print("  ├── tables/          (5 CSV files for paper)")
print("  ├── figures/         (4 PNG files, 300 DPI)")
print("  ├── translations/    (All model outputs)")
print("  ├── error_analysis/  (Detailed JSON files)")
print("  └── all_results.json (Complete metrics)")

print("\n" + "="*80)
print("🎉 READY FOR PAPER WRITING! 🎉")
print("="*80)
print("\nAll tables and figures are publication-ready (300 DPI).")
print("Find them in your Google Drive at:")
print(f"{DRIVE_BASE}/results/")
print("="*80)